# Study 903 — Sector-Neutral Low-Vol 🧮

**Is the low-vol anomaly a real stock-level effect, or just a bet on defensive sectors?**

The low-volatility anomaly (study 330) says calm stocks out-earn wild ones risk-adjusted.
But a naive low-vol sort quietly loads the structurally calm **sectors** — utilities,
staples, health care — and shorts the wild ones (tech, energy). So how much of the "edge"
is a defensive-**sector** bet rather than a stock-level effect? We strip the sector out:
rank each name on trailing volatility **within its own sector** (demean by the sector
median), then long the low-vol / short the high-vol names **sector-neutrally**, on a liquid
US cross-section (2010-01-04 → 2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The sector bet, made visible

Before we touch returns, look at *what the naive low-vol sort actually buys*. The calmest 30% of names are overwhelmingly the defensive sectors; the wildest 30% are tech and energy. So a raw low-vol book is, mechanically, long-defensive / short-cyclical — a sector bet wearing a factor costume.

In [1]:
R = dict(raw_long_def=44.7, raw_short_def=5.2, raw_ls_def=39.5, univ_def=20.0,
         neu_long_def=13.9, neu_short_def=15.7, neu_ls_def=-1.8)
print('RAW low-vol sort   : long book %.1f%% defensive vs short %.1f%% '
      '(universe %.1f%%) -> long-short tilt %+.1f%%'
      % (R['raw_long_def'], R['raw_short_def'], R['univ_def'], R['raw_ls_def']))
print('SECTOR-NEUTRAL sort: long book %.1f%% defensive vs short %.1f%% '
      '-> long-short tilt %+.1f%% (bet removed)'
      % (R['neu_long_def'], R['neu_short_def'], R['neu_ls_def']))

RAW low-vol sort   : long book 44.7% defensive vs short 5.2% (universe 20.0%) -> long-short tilt +39.5%
SECTOR-NEUTRAL sort: long book 13.9% defensive vs short 15.7% -> long-short tilt -1.8% (bet removed)


## 2. Does the low-vol edge survive the strip?

Now the returns. We run the identical long-low-vol / short-high-vol book twice — once raw, once after demeaning each name's vol within its sector — and compare.

In [2]:
R = dict(raw_spread_bps=-4.74, raw_t_nw=-2.61, neu_spread_bps=-3.53, neu_t_nw=-2.67,
         raw_lo_sh=0.98, raw_hi_sh=0.98, neu_lo_sh=1.05, neu_hi_sh=1.07)
print('spread (low-vol minus high-vol):')
print('  RAW           : %+.2f bps/day  (NW t = %+.2f)' % (R['raw_spread_bps'], R['raw_t_nw']))
print('  SECTOR-NEUTRAL: %+.2f bps/day  (NW t = %+.2f)' % (R['neu_spread_bps'], R['neu_t_nw']))
print()
print('per-leg Sharpe (the real, risk-adjusted low-vol claim):')
print('  RAW           : low-vol %.2f vs high-vol %.2f  (a tie)' % (R['raw_lo_sh'], R['raw_hi_sh']))
print('  SECTOR-NEUTRAL: low-vol %.2f vs high-vol %.2f  (high-vol edges it)' % (R['neu_lo_sh'], R['neu_hi_sh']))

spread (low-vol minus high-vol):
  RAW           : -4.74 bps/day  (NW t = -2.61)
  SECTOR-NEUTRAL: -3.53 bps/day  (NW t = -2.67)

per-leg Sharpe (the real, risk-adjusted low-vol claim):
  RAW           : low-vol 0.98 vs high-vol 0.98  (a tie)
  SECTOR-NEUTRAL: low-vol 1.05 vs high-vol 1.07  (high-vol edges it)


## 3. Is the demean real? A live synthetic control

The whole method rests on the demean genuinely stripping the sector bet. We prove it in a seeded toy world: plant **only** a defensive-sector premium (no stock-level effect at all) and watch the *raw* sort fire while the *sector-neutral* sort stays silent. Then plant a real within-sector low-vol effect and watch the neutral sort recover it. No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import pandas as pd
from sn_lowvol import data, strategy as st
secmap = pd.Series(data.synthetic_sectors(40, 8))
# only a sector premium (edge=0): raw should fire, neutral should not
conf = data.synthetic_panel(edge=0.0, seed=903, n_assets=40, n_days=1500, n_sectors=8, sector_prem_ann=0.08)
rc = st.close_returns(conf)
raw_t = st.vol_stats(st.vol_spreads(rc, secmap, 63, 0.3, neutral=False))['t_nw']
neu_t = st.vol_stats(st.vol_spreads(rc, secmap, 63, 0.3, neutral=True))['t_nw']
print('CONFOUND (sector premium only): raw NW t = %+.2f (fooled) -> neutral NW t = %+.2f (silent)' % (raw_t, neu_t))
# a genuine within-sector low-vol effect: neutral should recover it
pl = data.synthetic_panel(edge=0.1, seed=903, n_assets=40, n_days=1500, n_sectors=8)
pt = st.vol_stats(st.vol_spreads(st.close_returns(pl), secmap, 63, 0.3, neutral=True))['t_nw']
print('PLANTED stock-level low-vol effect: neutral NW t = %+.2f (recovered)' % pt)

CONFOUND (sector premium only): raw NW t = +3.13 (fooled) -> neutral NW t = -0.53 (silent)


PLANTED stock-level low-vol effect: neutral NW t = +4.23 (recovered)


## 4. The honest verdict — the low-vol edge does *not* survive here

On this liquid mega-cap tape the low-vol book is **negative** — the *wild* names (tech mega-caps) out-earned the calm ones — both raw (**-4.74 bps/day**) and sector-neutral (**-3.53 bps/day**, NW *t* = **-2.67**). Stripping the sector bet shifts the spread by only **+1.20 bps** and leaves it significantly *wrong-signed* vs the anomaly. Even on the anomaly's own turf — risk-adjusted **Sharpe** — the low-vol leg has *no* advantage once sector-neutral (1.05 vs 1.07). The synthetic control confirms the machinery is sound (a pure sector premium fools the raw sort at *t* = +3.46 but the neutral sort stays silent), so this is a real null, not a bug. **Signal: None** (the low-vol edge is absent — and what character the raw sort had was largely a defensive-sector tilt), **Tradability: Mirage** (the book loses money at any cost).